# Did the Strategy Work Today?

Run this **after 3:30 PM IST** on any trading day.  
Everything is automatic — no manual input required.  

It will tell you:
- What signal fired today (if any)
- What trade would have been placed
- Whether it would have hit SL, Target, or 10:15 exit
- Estimated P&L for that trade

In [ ]:
# ── Cell 1: Imports & Config ───────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import date, datetime, timedelta
from datetime import time as dtime
from pathlib import Path
from scipy.stats import norm as _norm
import warnings
warnings.filterwarnings('ignore')

BASE        = Path.cwd()
SIGNALS_CSV = BASE / 'v2' / 'v2_reliable_signals.csv'

# Must match py_backtest.py and run_everyday.ipynb exactly
GAP_THRESHOLD        = 0.0015
GAP_LARGE_THRESHOLD  = 0.0050
VIX_RISING_THRESHOLD = 0.03
VIX_SPIKE_THRESHOLD  = 0.05
BASE_RATE            = 54.5
STOP_LOSS_PCT        = 0.40
PROFIT_TARGET_PCT    = 0.40
STRIKE_STEP          = 50
STRIKES_OTM          = 1
LOT_SIZE             = 75
BROKERAGE            = 80
RISK_FREE_RATE       = 0.065
W                    = 68

# ── Signal mode & lot caps — must match py_backtest.py ───────────────────────
SIGNAL_MODE   = "BEARISH_ONLY"   # "ALL" | "BEARISH_ONLY" | "BULLISH_ONLY"
DTE0_MAX_LOTS = 10               # max lots when DTE=0 (BS unreliable + wide spreads)
MAX_LOTS      = 20               # hard global cap on any single trade

print('Config loaded. Run Cell 2 to fetch today\'s data.')
print(f'  Signal mode : {SIGNAL_MODE}  |  DTE0 cap: {DTE0_MAX_LOTS}  |  Global cap: {MAX_LOTS}')

In [2]:
# ── Cell 2: Fetch all data automatically ──────────────────────────────────────
today = date.today()

if today.weekday() >= 5:
    raise SystemExit(f'Today is {today.strftime("%A")} — market is closed.')

now_hour = datetime.now().hour
if now_hour < 15:
    print(f'WARNING: It is {datetime.now().strftime("%H:%M")} IST — market may not have closed yet.')
    print('Results will be incomplete. Run after 3:30 PM for full simulation.')
    print()

start_hist = str(today - timedelta(days=12))
end_hist   = str(today + timedelta(days=1))

print(f'Fetching data for {today} ...', end=' ', flush=True)

# ── Global overnight data (same as run_everyday) ──────────────────────────────
TICKERS = {
    'SP500'    : '^GSPC',
    'SGX'      : 'NKD=F',
    'DAX'      : '^GDAXI',
    'VIX_US'   : '^VIX',
    'NIFTY'    : '^NSEI',
    'VIX_INDIA': '^INDIAVIX',
}

raw = {}
for name, ticker in TICKERS.items():
    try:
        df = yf.download(ticker, start=start_hist, end=end_hist, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        raw[name] = df
    except Exception as e:
        raw[name] = pd.DataFrame()
        print(f'\n  WARNING: {name} fetch failed: {e}')

# ── Today's NIFTY 1-minute intraday data ──────────────────────────────────────
try:
    nifty_1m = yf.download('^NSEI', period='1d', interval='1m', progress=False, auto_adjust=True)
    if isinstance(nifty_1m.columns, pd.MultiIndex):
        nifty_1m.columns = nifty_1m.columns.get_level_values(0)
    if nifty_1m.index.tz is not None:
        nifty_1m.index = nifty_1m.index.tz_convert('Asia/Kolkata')
    else:
        nifty_1m.index = nifty_1m.index.tz_localize('Asia/Kolkata')
except Exception as e:
    nifty_1m = pd.DataFrame()
    print(f'\n  WARNING: intraday fetch failed: {e}')

print('done.')

# ── Extract values ─────────────────────────────────────────────────────────────
MAX_STALE = 5

def _ret(name, before=today):
    df = raw[name]
    rows = df[df.index.date < before]
    if len(rows) < 2: return None
    if (before - rows.index[-1].date()).days > MAX_STALE: return None
    return float((rows['Close'].iloc[-1] - rows['Close'].iloc[-2]) / rows['Close'].iloc[-2])

def _close(name, before=today):
    df = raw[name]
    rows = df[df.index.date < before]
    if len(rows) == 0: return None
    if (before - rows.index[-1].date()).days > MAX_STALE: return None
    return float(rows['Close'].iloc[-1])

def _today_close(name):
    df = raw[name]
    rows = df[df.index.date == today]
    if len(rows) == 0: return None
    return float(rows['Close'].iloc[-1])

sp500_ret    = _ret('SP500')
sgx_ret      = _ret('SGX')
dax_ret      = _ret('DAX')
vix_ret      = _ret('VIX_US')
nifty_ret    = _ret('NIFTY')
nifty_close  = _close('NIFTY')         # prev close
vix_india    = _today_close('VIX_INDIA') or _close('VIX_INDIA')  # today's India VIX

# Today's NIFTY open from 1-min data
nifty_open_today = None
nifty_close_today = None
first_hour_ret = None

if len(nifty_1m) > 0:
    open_candle = nifty_1m.between_time('09:15', '09:16')
    if len(open_candle) > 0:
        nifty_open_today = float(open_candle['Open'].iloc[0])
    close_candle = nifty_1m.between_time('10:14', '10:16')
    if len(close_candle) > 0:
        nifty_close_today = float(close_candle['Close'].iloc[-1])
    if nifty_open_today and nifty_close_today:
        first_hour_ret = (nifty_close_today - nifty_open_today) / nifty_open_today

# ── Print summary ──────────────────────────────────────────────────────────────
if nifty_close is None:
    raise SystemExit('NIFTY previous close unavailable. Possible market holiday.')

print(f'\n  NIFTY prev close   : {nifty_close:>10,.1f}')
print(f'  NIFTY today open   : {nifty_open_today:>10,.1f}' if nifty_open_today else '  NIFTY today open   :    MISSING')
print(f'  NIFTY 10:15 price  : {nifty_close_today:>10,.1f}' if nifty_close_today else '  NIFTY 10:15 price  :    MISSING')
print(f'  First hour return  : {first_hour_ret:>+10.2%}' if first_hour_ret is not None else '  First hour return  :    MISSING')
print(f'  India VIX          : {vix_india:>10.1f}' if vix_india else '  India VIX          :    MISSING')
print(f'  S&P 500 yesterday  : {sp500_ret:>+10.2%}' if sp500_ret is not None else '  S&P 500 yesterday  :    MISSING')
print(f'  SGX/Nikkei futures : {sgx_ret:>+10.2%}'   if sgx_ret   is not None else '  SGX/Nikkei futures :    MISSING')
print(f'  DAX yesterday      : {dax_ret:>+10.2%}'   if dax_ret   is not None else '  DAX yesterday      :    MISSING')
print(f'  VIX yesterday      : {vix_ret:>+10.2%}'   if vix_ret   is not None else '  VIX yesterday      :    MISSING')

Results will be incomplete. Run after 3:30 PM for full simulation.

Fetching data for 2026-04-02 ... done.

  NIFTY prev close   :   22,679.4
  NIFTY today open   :   22,383.4
  NIFTY 10:15 price  :    MISSING
  First hour return  :    MISSING
  India VIX          :       26.1
  S&P 500 yesterday  :     +0.72%
  SGX/Nikkei futures :     +2.19%
  DAX yesterday      :     +2.73%
  VIX yesterday      :     -2.81%


In [ ]:
# ── Cell 3: Compute signals — gap is now KNOWN ─────────────────────────────────
if nifty_open_today is None:
    raise SystemExit('NIFTY open price unavailable — cannot determine gap or signal.')

gap = (nifty_open_today - nifty_close) / nifty_close

signals = {
    'Gap Up'          : gap          >  GAP_THRESHOLD,
    'Gap Up Strong'   : gap          >  GAP_LARGE_THRESHOLD,
    'Gap Down'        : gap          < -GAP_THRESHOLD,
    'Prev India UP'   : nifty_ret    is not None and nifty_ret  > 0,
    'Prev India DOWN' : nifty_ret    is not None and nifty_ret  < 0,
    'US UP'           : sp500_ret    is not None and sp500_ret  > 0,
    'US DOWN'         : sp500_ret    is not None and sp500_ret  < 0,
    'SGX UP'          : sgx_ret      is not None and sgx_ret    > 0,
    'SGX DOWN'        : sgx_ret      is not None and sgx_ret    < 0,
    'DAX UP'          : dax_ret      is not None and dax_ret    > 0,
    'VIX Rising'      : vix_ret      is not None and vix_ret    > VIX_RISING_THRESHOLD,
    'VIX Falling'     : vix_ret      is not None and vix_ret    < 0,
    'VIX Spike'       : vix_ret      is not None and vix_ret    > VIX_SPIKE_THRESHOLD,
}

# Gap label
if gap > GAP_LARGE_THRESHOLD:  gap_label = f'GAP UP STRONG ({gap:+.2%})'
elif gap > GAP_THRESHOLD:      gap_label = f'GAP UP ({gap:+.2%})'
elif gap < -GAP_THRESHOLD:     gap_label = f'GAP DOWN ({gap:+.2%})'
else:                          gap_label = f'FLAT ({gap:+.2%})'

active = [k for k, v in signals.items() if v]
print(f'Gap         : {nifty_open_today:,.0f} vs prev close {nifty_close:,.0f}  ->  {gap_label}')
print(f'Active      : {chr(10).join("  + " + s for s in active) if active else "  (none)"}')

# ── Load top-3 bearish + bullish ──────────────────────────────────────────────
if not SIGNALS_CSV.exists():
    raise FileNotFoundError(f'{SIGNALS_CSV} not found.')

reliable    = pd.read_csv(SIGNALS_CSV)
top_bearish = reliable[reliable['P_Down'] > BASE_RATE].sort_values('Edge', ascending=False).head(3).reset_index(drop=True)
top_bullish = reliable[reliable['P_Down'] < (100 - BASE_RATE)].sort_values('P_Down').head(3).reset_index(drop=True)
bear_combos = list(top_bearish['Signal'])
bull_combos = list(top_bullish['Signal'])

def _fires(combo_str):
    return all(signals.get(s.strip(), False) for s in combo_str.split('+'))

bear_fired = [c for c in bear_combos if _fires(c)] if SIGNAL_MODE != "BULLISH_ONLY" else []
bull_fired = [c for c in bull_combos if _fires(c)] if SIGNAL_MODE != "BEARISH_ONLY" else []

print()
print('Top-3 bearish:')
for c in bear_combos:
    tag = 'FIRED' if c in bear_fired else ('-----' if SIGNAL_MODE == 'BULLISH_ONLY' else '     ')
    print(f'  {tag}  {c}')
print('Top-3 bullish:')
for c in bull_combos:
    tag = 'FIRED' if c in bull_fired else ('-----' if SIGNAL_MODE == 'BEARISH_ONLY' else '     ')
    print(f'  {tag}  {c}')
if SIGNAL_MODE != 'ALL':
    print(f'  [{"Bearish" if SIGNAL_MODE == "BULLISH_ONLY" else "Bullish"} combos suppressed — mode: {SIGNAL_MODE}]')

print()
print('=' * W)
if bear_fired and bull_fired:
    action = 'CONFLICT'
    print(f'  SIGNAL: CONFLICT — no trade (both bearish and bullish fired)')
elif bear_fired:
    action = 'BEARISH'
    r = top_bearish[top_bearish['Signal'] == bear_fired[0]].iloc[0]
    print(f'  SIGNAL: BEARISH  (would buy PUT)')
    print(f'  Combo : {bear_fired[0]}')
    print(f'  P(DOWN)={r["P_Down"]:.1f}%  Edge=+{r["Edge"]:.1f}%  N={int(r["N"])}')
elif bull_fired:
    action = 'BULLISH'
    r = top_bullish[top_bullish['Signal'] == bull_fired[0]].iloc[0]
    print(f'  SIGNAL: BULLISH  (would buy CALL)')
    print(f'  Combo : {bull_fired[0]}')
    print(f'  P(UP)={100-r["P_Down"]:.1f}%  Edge=+{abs(r["Edge"]):.1f}%  N={int(r["N"])}')
else:
    action = 'NO_SIGNAL'
    print(f'  SIGNAL: NEUTRAL — no trade today')
print('=' * W)

In [ ]:
# ── Cell 4: Simulate the trade using actual today's minute data ───────────────
if action not in ('BEARISH', 'BULLISH'):
    print(f'No trade today ({action}). Nothing to simulate.')
else:
    # ── Black-Scholes pricing ──────────────────────────────────────────────────
    def bs_price(S, K, T, r, sigma, opt_type='CE'):
        if T <= 1e-7:
            return max(0.0, S-K) if opt_type=='CE' else max(0.0, K-S)
        sq = sigma * np.sqrt(T)
        d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / sq
        d2 = d1 - sq
        if opt_type == 'CE':
            return float(S*_norm.cdf(d1) - K*np.exp(-r*T)*_norm.cdf(d2))
        return float(K*np.exp(-r*T)*_norm.cdf(-d2) - S*_norm.cdf(-d1))

    def nearest_thursday(d):
        days = (3 - d.weekday()) % 7
        return d + timedelta(days=days)

    # ── Trade setup ───────────────────────────────────────────────────────────
    expiry   = nearest_thursday(today)
    dte      = (expiry - today).days
    atm      = round(nifty_open_today / STRIKE_STEP) * STRIKE_STEP
    opt_type = 'PE' if action == 'BEARISH' else 'CE'
    strike   = atm - STRIKE_STEP*STRIKES_OTM if action == 'BEARISH' else atm + STRIKE_STEP*STRIKES_OTM

    iv_raw = (vix_india / 100.0) if vix_india else 0.15
    iv     = iv_raw * (1.30 if dte<=2 else 1.15 if dte<=4 else 1.05)

    def tte(t_obj):
        expiry_dt  = datetime.combine(expiry, dtime(15, 30))
        current_dt = datetime.combine(today, t_obj)
        secs = (expiry_dt - current_dt).total_seconds()
        return max(secs, 0.0) / (365.25 * 24 * 3600)

    entry_price = bs_price(nifty_open_today, strike, tte(dtime(9,15)), RISK_FREE_RATE, iv, opt_type)

    # ── Lot sizing with DTE caps (mirrors py_backtest.py) ─────────────────────
    FIXED_LOTS   = 5
    cost_per_lot = entry_price * LOT_SIZE
    # No live capital in this notebook; use fixed floor of 5 lots
    dte0_limited = min(FIXED_LOTS, DTE0_MAX_LOTS) if dte == 0 else FIXED_LOTS
    lots         = min(dte0_limited, MAX_LOTS)

    if entry_price < 0.5:
        print('Option is degenerate (price < 0.5 pts). No valid trade.')
    else:
        sl_price = entry_price * (1 - STOP_LOSS_PCT)
        tp_price = entry_price * (1 + PROFIT_TARGET_PCT)

        print(f'Trade setup:')
        print(f'  Instrument : NIFTY {today.strftime("%d%b%Y").upper()} {strike} {opt_type}')
        print(f'  Expiry     : {expiry}  (DTE={dte})')
        print(f'  Entry      : {entry_price:.1f} pts  (at 9:15 AM open)')
        print(f'  Stop Loss  : {sl_price:.1f} pts  ({STOP_LOSS_PCT:.0%} below entry)')
        print(f'  Target     : {tp_price:.1f} pts  ({PROFIT_TARGET_PCT:.0%} above entry)')
        print(f'  India VIX  : {vix_india:.1f}  ->  IV used = {iv*100:.1f}%')
        print()

        # ── Minute-by-minute simulation ───────────────────────────────────────
        first_hour = nifty_1m.between_time('09:16', '10:15') if len(nifty_1m) > 0 else pd.DataFrame()

        exit_price  = None
        exit_reason = '10:15 exit'
        exit_time   = '10:15'
        exit_nifty  = None

        for ts, row in first_hour.iterrows():
            spot  = float(row['Close'])
            t_now = ts.to_pydatetime().replace(tzinfo=None).time()
            price = bs_price(spot, strike, tte(t_now), RISK_FREE_RATE, iv, opt_type)
            if price <= sl_price:
                exit_price, exit_reason, exit_time, exit_nifty = sl_price, 'Stop Loss hit', str(t_now)[:5], spot
                break
            if price >= tp_price:
                exit_price, exit_reason, exit_time, exit_nifty = tp_price, 'Target hit', str(t_now)[:5], spot
                break

        if exit_price is None:
            if len(first_hour) > 0:
                spot_last = float(first_hour.iloc[-1]['Close'])
                exit_nifty = spot_last
                exit_price = bs_price(spot_last, strike, tte(dtime(10,15)), RISK_FREE_RATE, iv, opt_type)
            else:
                exit_price = entry_price  # fallback

        pnl_pts = exit_price - entry_price
        pnl_rs  = pnl_pts * LOT_SIZE * lots - BROKERAGE * lots
        outcome = 'PROFIT' if pnl_rs > 0 else 'LOSS'

        # ── Actual first-hour direction ────────────────────────────────────────
        actual_dir = None
        if first_hour_ret is not None:
            actual_dir = 'DOWN' if first_hour_ret < 0 else 'UP'
        predicted  = 'DOWN' if action == 'BEARISH' else 'UP'
        correct    = (actual_dir == predicted) if actual_dir else None

        # ── Results ───────────────────────────────────────────────────────────
        print('=' * W)
        print(f'  RESULT: {outcome}  ({exit_reason} at {exit_time})')
        print('=' * W)
        print(f'  Entry price : {entry_price:.1f} pts  (Rs {entry_price*LOT_SIZE:,.0f} per lot)')
        print(f'  Exit price  : {exit_price:.1f} pts  at {exit_time}')
        if exit_nifty:
            print(f'  NIFTY at exit: {exit_nifty:,.1f}')
        print()
        print(f'  Lots used   : {lots}  (cost/lot Rs {cost_per_lot:,.0f})')
        print(f'  P&L (pts)   : {pnl_pts:+.1f} pts  per lot')
        print(f'  P&L (Rs)    : Rs {pnl_rs:+,.0f}  ({lots} lot{"s" if lots > 1 else ""}, after brokerage)')
        print()
        if actual_dir:
            print(f'  Predicted   : {predicted}')
            print(f'  Actual      : {actual_dir}  ({first_hour_ret:+.2%} in first hour)')
            print(f'  Correct?    : {"YES" if correct else "NO"}')
        else:
            print(f'  Predicted   : {predicted}')
            print(f'  Actual dir  : unavailable (run after 10:15 AM)')
        print('=' * W)